In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv("dataset/Netfix_analysis.csv")

In [2]:
df.head(5)

,Release_Date,Title,Overview,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Poster_Url
0,2021-12-15,Spider-Man: No Way Home,Peter Parker is unmasked and no longer able to...,5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",https://image.tmdb.org/t/p/original/1g0dhYtq4i...
1,2022-03-01,The Batman,"In his second year of fighting crime, Batman u...",3827.658,1151,8.1,en,"Crime, Mystery, Thriller",https://image.tmdb.org/t/p/original/74xTEgt7R3...
2,2022-02-25,No Exit,Stranded at a rest stop in the mountains durin...,2618.087,122,6.3,en,Thriller,https://image.tmdb.org/t/p/original/vDHsLnOWKl...
3,2021-11-24,Encanto,"The tale of an extraordinary family, the Madri...",2402.201,5076,7.7,en,"Animation, Comedy, Family, Fantasy",https://image.tmdb.org/t/p/original/4j0PNHkMr5...
4,2021-12-22,The King's Man,As a collection of history's worst tyrants and...,1895.511,1793,7.0,en,"Action, Adventure, Thriller, War",https://image.tmdb.org/t/p/original/aq4Pwv5Xeu...


In [3]:
df = df.drop(columns=['Overview','Poster_Url'])

In [4]:
df.isnull().sum()

Release_Date         0
Title                0
Popularity           0
Vote_Count           0
Vote_Average         0
Original_Language    0
Genre                0
dtype: int64

In [5]:
df.groupby("Genre").count()

,Release_Date,Title,Popularity,Vote_Count,Vote_Average,Original_Language
Genre,,,,,,
Action,99,99,99,99,99,99
"Action, Adventure",24,24,24,24,24,24
"Action, Adventure, Animation",6,6,6,6,6,6
"Action, Adventure, Animation, Comedy",1,1,1,1,1,1
"Action, Adventure, Animation, Comedy, Family",3,3,3,3,3,3
...,...,...,...,...,...,...
"Western, Drama, Mystery",1,1,1,1,1,1
"Western, History",1,1,1,1,1,1
"Western, Horror",2,2,2,2,2,2


In [6]:
unique_genres = sorted(
    list(
        set(
            [
                g.strip()
                for sublist in df["Genre"].dropna().str.split(",")
                for g in sublist
            ]
        )
    )
)

print(f"Found {len(unique_genres)} unique genres in the file:")
print(unique_genres)

# 3. Create a brand new column for every discovered genre
for genre in unique_genres:
    # Checks if the text exists in the row's Genre entry
    df[genre] = np.where(
        df["Genre"].str.contains(genre, case=False, na=False), "Yes", "No"
    )

Found 19 unique genres in the file:
['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War', 'Western']


In [7]:
df.head(5)

,Release_Date,Title,Popularity,Vote_Count,Vote_Average,Original_Language,Genre,Action,Adventure,Animation,...,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western
0,2021-12-15,Spider-Man: No Way Home,5083.954,8940,8.3,en,"Action, Adventure, Science Fiction",Yes,Yes,No,...,No,No,No,No,No,Yes,No,No,No,No
1,2022-03-01,The Batman,3827.658,1151,8.1,en,"Crime, Mystery, Thriller",No,No,No,...,No,No,No,Yes,No,No,No,Yes,No,No
2,2022-02-25,No Exit,2618.087,122,6.3,en,Thriller,No,No,No,...,No,No,No,No,No,No,No,Yes,No,No
3,2021-11-24,Encanto,2402.201,5076,7.7,en,"Animation, Comedy, Family, Fantasy",No,No,Yes,...,No,No,No,No,No,No,No,No,No,No
4,2021-12-22,The King's Man,1895.511,1793,7.0,en,"Action, Adventure, Thriller, War",Yes,Yes,No,...,No,No,No,No,No,No,No,Yes,Yes,No


In [10]:
df.columns = df.columns.str.lower()

In [11]:


from sqlalchemy import create_engine

# Step 1: Connect to PostgreSQL
# Replace placeholders with your actual details
username = "postgres"      # default user
password = "2096" # the password you set during installation
host = "localhost"         # if running locally
port = "5432"              # default PostgreSQL port
database = "Netfix_Analysis"    # the database you created in pgAdmin

engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")

# Step 2: Load DataFrame into PostgreSQL
table_name = "movies"   # choose any table name
df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")

Data successfully loaded into table 'movies' in database 'Netfix_Analysis'.
